# Financial Evaluation of Fixed Hedging Outputs

This notebook performs the financial evaluation of the hedging strategies generated by the modelling stage.

Its purpose is to work from fixed outputs rather than from the training code itself. The notebook therefore focuses on the quantities that matter for the financial analysis:
the hedging error, the gross PnL, the net PnL, transaction costs, turnover, and a compact comparative table.

The baseline result is the strategy associated with the mean squared loss. If available, the strategy associated with the asymmetric loss is also evaluated. The workflow is intentionally independent from the modelling notebook so that the analysis can be reproduced, extended and discussed in a transparent way.


## Files expected in the current working tree

The notebook is written to match the current project structure.

Required files:
- `dataset (1).pkl`
- `config.json`
- `deltas_mse.npy`

Optional file:
- `deltas_asymmetric.npy`

The notebook first searches for these files in the project root. It also contains a fallback recursive search so that it remains robust to small changes in the directory structure.


In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## File detection

The first step is to identify the local files that contain the dataset, the configuration and the fixed hedging strategies. The detection logic is explicit in order to make the notebook easy to reuse.


In [ ]:
PROJECT_ROOT = Path(".").resolve()
RESULTS_DIR = PROJECT_ROOT / "person2_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def find_file(preferred_name, fallback_pattern=None, start_dir=PROJECT_ROOT, required=True):
    preferred_path = start_dir / preferred_name
    if preferred_path.exists():
        return preferred_path

    patterns = []
    if fallback_pattern is not None:
        patterns.append(fallback_pattern)
    patterns.append(preferred_name)

    matches = []
    for pattern in patterns:
        matches.extend(start_dir.rglob(pattern))

    # remove duplicates while preserving order
    unique_matches = []
    seen = set()
    for m in matches:
        key = str(m.resolve())
        if key not in seen:
            seen.add(key)
            unique_matches.append(m)

    if not unique_matches:
        if required:
            raise FileNotFoundError(f"Could not find file '{preferred_name}' from {start_dir}")
        return None

    if len(unique_matches) > 1:
        print(f"Multiple matches found for {preferred_name}:")
        for m in unique_matches:
            print(" -", m)
        print("Using:", unique_matches[0])

    return unique_matches[0]


dataset_file = find_file("dataset (1).pkl", fallback_pattern="dataset*.pkl")
config_file = find_file("config.json")
baseline_file = find_file("deltas_mse.npy")
asymmetric_file = find_file("deltas_asymmetric.npy", required=False)

print("Dataset file      :", dataset_file)
print("Config file       :", config_file)
print("Baseline deltas   :", baseline_file)
print("Asymmetric deltas :", asymmetric_file)
print("Results directory :", RESULTS_DIR)


## Load the dataset and configuration

The dataset contains the simulated paths produced in Phase 1.
The test set is the relevant subset for the present analysis because the objective is to evaluate the hedging strategy out of sample.


In [ ]:
with open(dataset_file, "rb") as f:
    dataset = pickle.load(f)

with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)

time_days = np.array(dataset["time_days"])
test_state_paths = np.array(dataset["test"]["state_paths"])
test_tradable_paths = np.array(dataset["test"]["tradable_paths"])
test_volume_paths = np.array(dataset["test"]["volume_paths"])
test_payoff = np.array(dataset["test"]["payoff"])

liquidity = np.array(config["liquidity"], dtype=float)
transaction_costs = np.array(config["transaction_costs"], dtype=float)

print("time grid shape       :", time_days.shape)
print("test state shape      :", test_state_paths.shape)
print("test tradable shape   :", test_tradable_paths.shape)
print("test volume shape     :", test_volume_paths.shape)
print("test payoff shape     :", test_payoff.shape)
print("liquidity vector      :", liquidity)
print("transaction costs     :", transaction_costs)


## Load fixed hedging outputs

The baseline corresponds to the strategy trained under the mean squared loss. The asymmetric-loss strategy is included if the corresponding file is available.


In [ ]:
hedges_by_model = {
    "baseline_mse": np.load(baseline_file)
}

if asymmetric_file is not None:
    hedges_by_model["asymmetric_loss"] = np.load(asymmetric_file)

for name, hedges in hedges_by_model.items():
    print(name, hedges.shape)


## Validation checks

Before computing financial metrics, the notebook checks two structural properties.

First, the shape of the hedging matrix must match the test paths.  
Second, the liquidity constraints must be respected by the positions and by the position increments.

These checks are useful because they connect the machine learning output to the economic constraints of the problem.


In [ ]:
def validate_hedges_shape(tradable_paths, hedges):
    n_paths, n_steps_plus_one, d = tradable_paths.shape
    expected_shape = (n_paths, n_steps_plus_one - 1, d)
    if hedges.shape != expected_shape:
        raise ValueError(f"Expected hedges shape {expected_shape}, got {hedges.shape}")


def check_liquidity_constraints(hedges, liquidity):
    liquidity = np.asarray(liquidity, dtype=float)
    if np.any(np.abs(hedges[:, 0, :]) > liquidity + 1e-12):
        raise ValueError("Initial hedge positions violate liquidity constraints.")
    increments = np.diff(hedges, axis=1)
    if np.any(np.abs(increments) > liquidity + 1e-12):
        raise ValueError("Hedge increments violate liquidity constraints.")


for name, hedges in hedges_by_model.items():
    validate_hedges_shape(test_tradable_paths, hedges)
    check_liquidity_constraints(hedges, liquidity)
    print(f"{name}: shape and liquidity constraints validated")


## Financial evaluation functions

The financial evaluation is organized around five core quantities.

The terminal portfolio value measures the cumulative self-financing gains.  
The hedging error compares the terminal portfolio value to the derivative payoff.  
The gross PnL is equal to the hedging error in the present setting because no premium is introduced here.  
The net PnL adjusts the gross PnL for transaction costs.  
Turnover measures the total trading intensity and is useful to interpret implementation costs.


In [ ]:
def compute_terminal_portfolio_value(tradable_paths, hedges, premium=0.0):
    price_increments = tradable_paths[:, 1:, :] - tradable_paths[:, :-1, :]
    gains = np.sum(hedges * price_increments, axis=(1, 2))
    return premium + gains


def compute_transaction_costs(hedges, transaction_costs):
    transaction_costs = np.asarray(transaction_costs, dtype=float)
    first_trade = np.abs(hedges[:, 0, :])
    later_trades = np.abs(np.diff(hedges, axis=1))
    all_trades = np.concatenate([first_trade[:, None, :], later_trades], axis=1)
    return np.sum(all_trades * transaction_costs[None, None, :], axis=(1, 2))


def compute_hedging_error(tradable_paths, hedges, payoff, premium=0.0):
    terminal_value = compute_terminal_portfolio_value(tradable_paths, hedges, premium=premium)
    return terminal_value - payoff


def compute_gross_pnl(tradable_paths, hedges, payoff, premium=0.0):
    return compute_terminal_portfolio_value(tradable_paths, hedges, premium=premium) - payoff


def compute_net_pnl(tradable_paths, hedges, payoff, premium=0.0, transaction_costs=None):
    gross_pnl = compute_gross_pnl(tradable_paths, hedges, payoff, premium=premium)
    if transaction_costs is None:
        return gross_pnl
    costs = compute_transaction_costs(hedges, transaction_costs)
    return gross_pnl - costs


def compute_turnover(hedges):
    first_trade = np.abs(hedges[:, 0, :])
    later_trades = np.abs(np.diff(hedges, axis=1))
    all_trades = np.concatenate([first_trade[:, None, :], later_trades], axis=1)
    return np.sum(all_trades, axis=(1, 2))


def summarize_distribution(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "q01": float(np.quantile(x, 0.01)),
        "q05": float(np.quantile(x, 0.05)),
        "median": float(np.median(x)),
        "q95": float(np.quantile(x, 0.95)),
        "q99": float(np.quantile(x, 0.99)),
        "max": float(np.max(x)),
    }


def empirical_cvar_left(x, alpha=0.05):
    threshold = np.quantile(x, alpha)
    tail = x[x <= threshold]
    return float(np.mean(tail))


## Compute the financial outputs

This block applies the evaluation pipeline to all fixed strategies currently available.
The resulting objects are stored in memory and then saved to disk for reproducibility.


In [ ]:
financial_outputs = {}
summary_rows = []

for model_name, hedges in hedges_by_model.items():
    terminal_value = compute_terminal_portfolio_value(test_tradable_paths, hedges, premium=0.0)
    hedging_error = compute_hedging_error(test_tradable_paths, hedges, test_payoff, premium=0.0)
    gross_pnl = compute_gross_pnl(test_tradable_paths, hedges, test_payoff, premium=0.0)
    net_pnl = compute_net_pnl(
        test_tradable_paths,
        hedges,
        test_payoff,
        premium=0.0,
        transaction_costs=transaction_costs
    )
    costs = compute_transaction_costs(hedges, transaction_costs)
    turnover = compute_turnover(hedges)

    financial_outputs[model_name] = {
        "terminal_value": terminal_value,
        "hedging_error": hedging_error,
        "gross_pnl": gross_pnl,
        "net_pnl": net_pnl,
        "transaction_costs": costs,
        "turnover": turnover,
        "hedges": hedges,
    }

    summary_rows.append({
        "model": model_name,
        "mse_hedging_error": float(np.mean(hedging_error ** 2)),
        "mean_hedging_error": float(np.mean(hedging_error)),
        "std_hedging_error": float(np.std(hedging_error)),
        "gross_pnl_mean": float(np.mean(gross_pnl)),
        "gross_pnl_std": float(np.std(gross_pnl)),
        "net_pnl_mean": float(np.mean(net_pnl)),
        "net_pnl_std": float(np.std(net_pnl)),
        "net_pnl_q05": float(np.quantile(net_pnl, 0.05)),
        "net_pnl_cvar05": empirical_cvar_left(net_pnl, alpha=0.05),
        "mean_transaction_cost": float(np.mean(costs)),
        "mean_turnover": float(np.mean(turnover)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("model").reset_index(drop=True)
summary_df


## Save the numerical outputs

This step preserves the quantities that are useful for the report and for any later checks.


In [ ]:
for model_name, outputs in financial_outputs.items():
    np.save(RESULTS_DIR / f"{model_name}_hedging_error.npy", outputs["hedging_error"])
    np.save(RESULTS_DIR / f"{model_name}_gross_pnl.npy", outputs["gross_pnl"])
    np.save(RESULTS_DIR / f"{model_name}_net_pnl.npy", outputs["net_pnl"])
    np.save(RESULTS_DIR / f"{model_name}_transaction_costs.npy", outputs["transaction_costs"])
    np.save(RESULTS_DIR / f"{model_name}_turnover.npy", outputs["turnover"])

summary_df.to_csv(RESULTS_DIR / "financial_summary_table_full.csv", index=False)
summary_df.to_json(RESULTS_DIR / "financial_summary_table_full.json", orient="records", indent=2)

print("Saved outputs in:", RESULTS_DIR.resolve())


## Main empirical results

The table below is the compact version intended for the report.
It keeps the metrics that are the most meaningful from a financial point of view:
residual hedging risk, downside risk, transaction costs and turnover.


In [ ]:
labels = {
    "baseline_mse": "Baseline (MSE)",
    "asymmetric_loss": "Asymmetric loss"
}

report_table = summary_df.copy()
report_table["model"] = report_table["model"].map(lambda x: labels.get(x, x))
report_table = report_table[[
    "model",
    "mse_hedging_error",
    "net_pnl_mean",
    "net_pnl_std",
    "net_pnl_q05",
    "net_pnl_cvar05",
    "mean_transaction_cost",
    "mean_turnover"
]]
report_table


In [ ]:
report_table.to_csv(RESULTS_DIR / "report_table.csv", index=False)
report_table.to_json(RESULTS_DIR / "report_table.json", orient="records", indent=2)


## Final figure selection

Only a small number of figures should be kept in the report.

The most useful ones are:
1. the distribution of the hedging error,
2. the distribution of the net PnL,
3. the average hedge paths.

These three figures summarize the quality of the hedge, the distribution of residual risk and the trading behaviour implied by the model.


In [ ]:
models = list(financial_outputs.keys())
display_names = {k: labels.get(k, k) for k in models}


In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4), squeeze=False)

for ax, model_name in zip(axes[0], models):
    ax.hist(financial_outputs[model_name]["hedging_error"], bins=60)
    ax.set_title(display_names[model_name])
    ax.set_xlabel("Hedging error")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figure_hedging_error_histograms.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4), squeeze=False)

for ax, model_name in zip(axes[0], models):
    ax.hist(financial_outputs[model_name]["net_pnl"], bins=60)
    ax.set_title(display_names[model_name])
    ax.set_xlabel("Net PnL")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figure_net_pnl_histograms.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4), squeeze=False)

for ax, model_name in zip(axes[0], models):
    mean_delta = financial_outputs[model_name]["hedges"].mean(axis=0)
    ax.plot(time_days[:-1], mean_delta[:, 0], label="Asset 1")
    ax.plot(time_days[:-1], mean_delta[:, 1], label="Asset 2")
    ax.set_title(display_names[model_name])
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Average hedge position")
    ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figure_average_hedge_paths.png", dpi=150)
plt.show()


## Structured interpretation

The text below is intentionally concise. It is written in a form that can be adapted directly into the report.

The interpretation focuses on four dimensions:
the residual hedging risk,
the downside tail risk,
the transaction-cost burden,
and the trading intensity.


In [ ]:
def build_interpretation(report_table):
    rows = []
    if len(report_table) == 1:
        row = report_table.iloc[0]
        rows.append(
            f"The baseline strategy achieves a hedging-error MSE of {row['mse_hedging_error']:.4f}. "
            f"Its average net PnL is {row['net_pnl_mean']:.4f}, with a standard deviation of {row['net_pnl_std']:.4f}."
        )
        rows.append(
            f"The 5% net-PnL quantile is {row['net_pnl_q05']:.4f}, and the associated left-tail CVaR is {row['net_pnl_cvar05']:.4f}. "
            f"These quantities summarize the residual downside risk after hedging."
        )
        rows.append(
            f"Average transaction costs are {row['mean_transaction_cost']:.4f}, for an average turnover of {row['mean_turnover']:.4f}. "
            f"This indicates the trading intensity required by the strategy."
        )
    else:
        base = report_table.iloc[0]
        alt = report_table.iloc[1]
        rows.append(
            f"The baseline strategy reaches a hedging-error MSE of {base['mse_hedging_error']:.4f}, "
            f"while the alternative specification reaches {alt['mse_hedging_error']:.4f}."
        )
        rows.append(
            f"In net PnL terms, the baseline has mean {base['net_pnl_mean']:.4f} and standard deviation {base['net_pnl_std']:.4f}, "
            f"against {alt['net_pnl_mean']:.4f} and {alt['net_pnl_std']:.4f} for the alternative."
        )
        rows.append(
            f"The 5% left-tail quantile is {base['net_pnl_q05']:.4f} for the baseline and {alt['net_pnl_q05']:.4f} for the alternative. "
            f"The corresponding CVaR values are {base['net_pnl_cvar05']:.4f} and {alt['net_pnl_cvar05']:.4f}."
        )
        rows.append(
            f"Average transaction costs are {base['mean_transaction_cost']:.4f} for the baseline and {alt['mean_transaction_cost']:.4f} for the alternative, "
            f"with turnover levels of {base['mean_turnover']:.4f} and {alt['mean_turnover']:.4f}, respectively."
        )
        rows.append(
            "The comparison should therefore not be based on the MSE alone. "
            "It should jointly consider residual variance, downside protection and the trading effort required to implement the strategy."
        )
    return "\n\n".join(rows)

interpretation_text = build_interpretation(report_table)
print(interpretation_text)


In [ ]:
with open(RESULTS_DIR / "structured_interpretation.txt", "w", encoding="utf-8") as f:
    f.write(interpretation_text)


## Main takeaways

At this stage, the notebook produces a complete financial evaluation for all fixed hedging outputs currently available.

The baseline result is sufficient to finalize the financial analysis of the baseline model.  
If the asymmetric-loss output is available, the notebook also supports the comparison associated with the loss-function modification.

If new strategies are produced later, the same pipeline can be reused by adding them to `hedges_by_model`.
